# NOTEARS Structure Learning on Simulated DAGs

This notebook evaluates **MDM with NOTEARS** (`method="notears"`) structure learning on the same simulated DAG structures as notebook 04:
- 3-variable DAG (Figura 5): Y1 -> Y2 -> Y3 (chain)
- 5-variable DAG (Figura 6): Y1->Y2, Y1->Y3, Y2->Y4, Y3->Y4, Y2->Y5

We compute evaluation metrics including:
- Connection accuracy
- Sensitivity (true positive rate)
- Specificity (true negative rate)
- Positive Predictive Value (PPV)
- Negative Predictive Value (NPV)
- Directional accuracy

In [1]:
import os
import sys
import time
from itertools import combinations

import numpy as np
import pandas as pd

# Add parent directory to path to import mdmp
sys.path.insert(0, os.path.abspath('../..'))
from mdmp import MDM

d:\dev\phd\sandbox\mdmr-python\mdmp\.venv-win\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def build_connection_matrix(adj_mat):
    """Build symmetric connection matrix from directed adjacency matrix."""
    n_n = adj_mat.shape[0]
    k = np.zeros((n_n, n_n))

    pairs = list(combinations(range(n_n), 2))
    m_con = np.zeros(len(pairs))
    for idx, (i, j) in enumerate(pairs):
        m_con[idx] = adj_mat[i, j] + adj_mat[j, i]

    for idx, (i, j) in enumerate(pairs):
        k[i, j] = k[j, i] = 1 if m_con[idx] == 1 else 0

    return {
        'connection_matrix': k,
        'connection_vector': m_con,
        'lower_ind': [(i, j) for i, j in pairs if i > j],
        'upper_ind': [(i, j) for i, j in pairs if i < j]
    }


def compute_metrics(true_adj, estimated_adj):
    """Compute evaluation metrics for DAG structure learning."""
    n_n = true_adj.shape[0]

    true_con = build_connection_matrix(true_adj)
    est_con = build_connection_matrix(estimated_adj)

    m_con_true = true_con['connection_vector']
    m_con_est = est_con['connection_vector']

    accuracy = np.mean(m_con_est == m_con_true)

    if np.any(m_con_true == 1):
        sensitivity = np.mean(m_con_est[m_con_true == 1] == 1)
    else:
        sensitivity = np.nan

    if np.any(m_con_true == 0):
        specificity = np.mean(m_con_est[m_con_true == 0] == 0)
    else:
        specificity = np.nan

    if np.any(m_con_est == 1):
        ppv = np.mean(m_con_true[m_con_est == 1] == 1)
    else:
        ppv = np.nan

    if np.any(m_con_est == 0):
        npv = np.mean(m_con_true[m_con_est == 0] == 0)
    else:
        npv = np.nan

    k = true_con['connection_matrix']
    pairs_with_connections = [(i, j) for i, j in combinations(range(n_n), 2) if k[i, j] == 1]

    if pairs_with_connections:
        d_accuracy_vals = [
            (true_adj[i, j] == estimated_adj[i, j]) &
            (true_adj[j, i] == estimated_adj[j, i])
            for i, j in pairs_with_connections
        ]
        d_accuracy = np.mean(d_accuracy_vals)
    else:
        d_accuracy = np.nan

    return {
        'accuracy': accuracy,
        'sensitivity': sensitivity,
        'specificity': specificity,
        'ppv': ppv,
        'npv': npv,
        'directional_accuracy': d_accuracy
    }

## Load Simulated Data

Load the simulated data files that were generated by `simulate_dags.py`.

In [3]:
data_dir = "./data/"

# Load 3-variable data (Figura 5)
data_3var = pd.read_csv(os.path.join(data_dir, "dag_3var_simulated.csv"))
true_adj_3var = pd.read_csv(
    os.path.join(data_dir, "dag_3var_true_adjacency.csv"),
    index_col=0
).values.astype(int)

# Load 5-variable data (Figura 6)
data_5var = pd.read_csv(os.path.join(data_dir, "dag_5var_simulated.csv"))
true_adj_5var = pd.read_csv(
    os.path.join(data_dir, "dag_5var_true_adjacency.csv"),
    index_col=0
).values.astype(int)

print("3-variable data shape:", data_3var.shape)
print("3-variable true adjacency shape:", true_adj_3var.shape)
print("\n5-variable data shape:", data_5var.shape)
print("5-variable true adjacency shape:", true_adj_5var.shape)

3-variable data shape: (200, 3)
3-variable true adjacency shape: (3, 3)

5-variable data shape: (200, 5)
5-variable true adjacency shape: (5, 5)


## 3-Variable DAG: NOTEARS Structure Learning

In [6]:
# Run MDM with NOTEARS on 3-variable data
print("Running MDM with NOTEARS on 3-variable DAG...")
start_time = time.time()

model_3var = MDM(data_3var, method="notears", nbf=15, verbose=False)

end_time = time.time()
time_3var = end_time - start_time

print(f"\nTime taken: {time_3var:.2f} seconds")
print("\nEstimated adjacency matrix:")
print(pd.DataFrame(
    model_3var.adj_mat,
    index=model_3var.node_names,
    columns=model_3var.node_names
))

Running MDM with NOTEARS on 3-variable DAG...

Time taken: 1.21 seconds

Estimated adjacency matrix:
    Y1  Y2  Y3
Y1   0   1   0
Y2   0   0   1
Y3   0   0   0


In [7]:
# Compute metrics for 3-variable DAG
metrics_3var = compute_metrics(true_adj_3var, model_3var.adj_mat)

print("Evaluation Metrics for 3-Variable DAG:")
print(f"  Accuracy: {metrics_3var['accuracy']:.4f}")
print(f"  Sensitivity: {metrics_3var['sensitivity']:.4f}")
print(f"  Specificity: {metrics_3var['specificity']:.4f}")
print(f"  PPV: {metrics_3var['ppv']:.4f}")
print(f"  NPV: {metrics_3var['npv']:.4f}")
print(f"  Directional Accuracy: {metrics_3var['directional_accuracy']:.4f}")

print(f"\nNumber of edges (estimated): {np.sum(model_3var.adj_mat)}")
print(f"Number of edges (true): {np.sum(true_adj_3var)}")

Evaluation Metrics for 3-Variable DAG:
  Accuracy: 1.0000
  Sensitivity: 1.0000
  Specificity: 1.0000
  PPV: 1.0000
  NPV: 1.0000
  Directional Accuracy: 1.0000

Number of edges (estimated): 2
Number of edges (true): 2


## 5-Variable DAG: NOTEARS Structure Learning

In [8]:
# Run MDM with NOTEARS on 5-variable data
print("Running MDM with NOTEARS on 5-variable DAG...")
start_time = time.time()

model_5var = MDM(data_5var, method="notears", nbf=15, verbose=False)

end_time = time.time()
time_5var = end_time - start_time

print(f"\nTime taken: {time_5var:.2f} seconds")
print("\nEstimated adjacency matrix:")
print(pd.DataFrame(
    model_5var.adj_mat,
    index=model_5var.node_names,
    columns=model_5var.node_names
))

Running MDM with NOTEARS on 5-variable DAG...

Time taken: 2.34 seconds

Estimated adjacency matrix:
    Y1  Y2  Y3  Y4  Y5
Y1   0   1   1   0   0
Y2   0   0   0   1   0
Y3   0   0   0   1   0
Y4   0   0   0   0   0
Y5   0   0   0   0   0


In [9]:
# Compute metrics for 5-variable DAG
metrics_5var = compute_metrics(true_adj_5var, model_5var.adj_mat)

print("Evaluation Metrics for 5-Variable DAG:")
print(f"  Accuracy: {metrics_5var['accuracy']:.4f}")
print(f"  Sensitivity: {metrics_5var['sensitivity']:.4f}")
print(f"  Specificity: {metrics_5var['specificity']:.4f}")
print(f"  PPV: {metrics_5var['ppv']:.4f}")
print(f"  NPV: {metrics_5var['npv']:.4f}")
print(f"  Directional Accuracy: {metrics_5var['directional_accuracy']:.4f}")

print(f"\nNumber of edges (estimated): {np.sum(model_5var.adj_mat)}")
print(f"Number of edges (true): {np.sum(true_adj_5var)}")

Evaluation Metrics for 5-Variable DAG:
  Accuracy: 0.9000
  Sensitivity: 0.8000
  Specificity: 1.0000
  PPV: 1.0000
  NPV: 0.8333
  Directional Accuracy: 0.8000

Number of edges (estimated): 4
Number of edges (true): 5


## Performance Summary

In [10]:
summary = pd.DataFrame({
    'DAG': ['3-var', '5-var'],
    'Accuracy': [metrics_3var['accuracy'], metrics_5var['accuracy']],
    'Sensitivity': [metrics_3var['sensitivity'], metrics_5var['sensitivity']],
    'Specificity': [metrics_3var['specificity'], metrics_5var['specificity']],
    'PPV': [metrics_3var['ppv'], metrics_5var['ppv']],
    'NPV': [metrics_3var['npv'], metrics_5var['npv']],
    'Dir.Acc': [metrics_3var['directional_accuracy'], metrics_5var['directional_accuracy']],
    'Edges_est': [np.sum(model_3var.adj_mat), np.sum(model_5var.adj_mat)],
    'Edges_true': [np.sum(true_adj_3var), np.sum(true_adj_5var)],
    'Time(s)': [time_3var, time_5var],
})
print("NOTEARS Performance Summary:")
print(summary.to_string(index=False))

NOTEARS Performance Summary:
  DAG  Accuracy  Sensitivity  Specificity  PPV      NPV  Dir.Acc  Edges_est  Edges_true  Time(s)
3-var       1.0          1.0          1.0  1.0 1.000000      1.0          2           2 1.213963
5-var       0.9          0.8          1.0  1.0 0.833333      0.8          4           5 2.337843


In [11]:
# Save results
results_3var = pd.DataFrame({
    'metric': ['accuracy', 'sensitivity', 'specificity', 'ppv', 'npv', 'directional_accuracy',
               'num_edges_estimated', 'num_edges_true', 'computation_time'],
    'value': [
        metrics_3var['accuracy'],
        metrics_3var['sensitivity'],
        metrics_3var['specificity'],
        metrics_3var['ppv'],
        metrics_3var['npv'],
        metrics_3var['directional_accuracy'],
        np.sum(model_3var.adj_mat),
        np.sum(true_adj_3var),
        time_3var
    ]
})
results_3var['dag'] = '3var'
results_3var['method'] = 'notears'

results_5var = pd.DataFrame({
    'metric': ['accuracy', 'sensitivity', 'specificity', 'ppv', 'npv', 'directional_accuracy',
               'num_edges_estimated', 'num_edges_true', 'computation_time'],
    'value': [
        metrics_5var['accuracy'],
        metrics_5var['sensitivity'],
        metrics_5var['specificity'],
        metrics_5var['ppv'],
        metrics_5var['npv'],
        metrics_5var['directional_accuracy'],
        np.sum(model_5var.adj_mat),
        np.sum(true_adj_5var),
        time_5var
    ]
})
results_5var['dag'] = '5var'
results_5var['method'] = 'notears'

results_notears = pd.concat([results_3var, results_5var], ignore_index=True)
results_notears.to_csv(os.path.join(data_dir, "notears_results.csv"), index=False)

pd.DataFrame(
    model_3var.adj_mat,
    index=model_3var.node_names,
    columns=model_3var.node_names
).to_csv(os.path.join(data_dir, "notears_adjacency_3var.csv"))
pd.DataFrame(
    model_5var.adj_mat,
    index=model_5var.node_names,
    columns=model_5var.node_names
).to_csv(os.path.join(data_dir, "notears_adjacency_5var.csv"))

print("Results saved to:", os.path.join(data_dir, "notears_results.csv"))

Results saved to: ./data/notears_results.csv
